# 04 — In-Sample Event Validation (Quick Sanity Check)

The in-sample sweep in notebook 03 promoted its winners to `backtest_results/top_selection` — but it only ever ran the **vector** engine, which abstracts away intra-bar order routing, partial fills, and cash constraints. Before spending the (slower) vector budget on two full out-of-sample regimes in notebook 05, it's worth a cheap sanity check: replay just the **top 10** in-sample winners with the **event-driven** engine on **only the first rolling window**. If a strategy's vector-engine edge evaporates the moment orders are routed bar-by-bar, it isn't worth carrying into out-of-sample testing at all.

This notebook is also the showcase for the framework's newest study-reuse API — the exact workflow this notebook needs (*"run a vector sweep, pick the top N, event-validate one window of the same study, merge the result back into the same bundle"*) used to require rebuilding the `Study` by hand. Now it's:

- **`get_backtests(storage_dir, algorithm_ids)`** — reload just the top 10 bundles by id (no need to rank/open the whole directory again).
- **`Backtest.get_study_definition(name)`** — pull the `in_sample_param_sweep` study straight off a loaded bundle (universe, windows, execution assumptions all carried over, `engine_results` reset) instead of re-declaring it.
- Slice `study.backtest_windows` down to the first window and swap `study.engines` to `EVENT_DRIVEN`.
- `app.run_backtest(..., backtest_storage_directory=top_selection_path)` merges the new `event` slot into the **same** `<algorithm_id>.obtf` bundle the vector sweep already wrote — the `vector` slot and its 20+ windows are untouched.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from strategies.supertrend_ema_confirmation.strategy import (
    SupertrendEmaConfirmationStrategy as Strategy,
)


## Constants


In [ ]:
from pathlib import Path

data_storage_path = Path.cwd().parent / "data"
backtest_results_dir = Path.cwd().parent / "backtest_results"
top_selection_path = backtest_results_dir / "top_selection"
reports_dir = Path.cwd().parent / "reports"
figures_dir = reports_dir / "figures"

IN_SAMPLE_STUDY = "in_sample_param_sweep"
TOP_N = 10


## Rank the in-sample sweep and pick the top 10

Same Tier-1 SQLite index + `rank_index` pattern as notebook 03's promotion step, just re-run here to pull out the top `TOP_N` `algorithm_id`s (the promoted `top_selection/` folder may hold more than 10 — e.g. notebook 03 promoted the top 20).


In [ ]:
from investing_algorithm_framework import BacktestEvaluationFocus, build_index, rank_index

# 1. (Re)build the Tier-1 SQLite index over the promoted top selection.
build_index(
    str(top_selection_path),
    show_progress=True,
    incremental=True,
)

# 2. Re-rank with the same BALANCED focus used to promote these bundles
#    in notebook 03, scoped to the in-sample study so a bundle that
#    later also carries OOS/event slots is never scored on a mix of
#    studies.
top_rows = rank_index(
    str(top_selection_path),
    focus=BacktestEvaluationFocus.BALANCED,
    engine="vector",
    study=IN_SAMPLE_STUDY,
    limit=TOP_N,
)

top_10_ids = [row["algorithm_id"] for row in top_rows]
print(f"Top {len(top_10_ids)} in-sample algorithm_ids (BALANCED focus, vector engine):")
for algorithm_id in top_10_ids:
    print(f"  {algorithm_id}")


## Reload just the top 10 by id, and recover the in-sample study

`get_backtests(storage_dir, algorithm_ids)` loads exactly the 10 bundles we just ranked — no need to reopen (or dedupe) the whole `top_selection/` folder via `LocalDirStore` the way earlier notebooks did. `get_backtest(storage_dir, algorithm_id)` (singular) does the same for one id — handy for pulling a single reference `Study` off any bundle in the set, since every bundle in this folder carries the identical `in_sample_param_sweep` definition.


In [ ]:
from investing_algorithm_framework import BacktestEngine, get_backtest, get_backtests

# Reload exactly the 10 ranked bundles by id.
top_10_backtests = get_backtests(str(top_selection_path), top_10_ids)
print(f"Reloaded {len(top_10_backtests)} of {len(top_10_ids)} requested backtests.")

# Pull the in-sample study straight off one of them -- a
# ``copy_definition()`` clone: universe, windows and execution
# assumptions carried over, ``engine_results`` reset to empty so it's
# ready for a fresh run.
reference_backtest = get_backtest(str(top_selection_path), top_10_ids[0])
event_study = reference_backtest.get_study_definition(IN_SAMPLE_STUDY)
assert event_study is not None, f"{IN_SAMPLE_STUDY!r} not found on {top_10_ids[0]}"
assert event_study.get_runs("vector") == [], "copy_definition() should reset engine_results"

print(f"\nRecovered study {event_study.name!r} with {len(event_study.backtest_windows)} windows.")

# Restrict to only the FIRST rolling window, switch to the event engine.
event_study.backtest_windows = event_study.backtest_windows[:1]
event_study.engines = [BacktestEngine.EVENT_DRIVEN]

print(
    f"Running only window 1 ({event_study.backtest_windows[0]}) "
    f"on the {event_study.engines[0]} engine."
)


## Re-instantiate the top 10 winning strategies

Each bundle's original grid variant was recorded under `Backtest.metadata["params"]` when notebook 03 built the strategy (see `initialize_strategies` there). We recover it here and rebuild the same `algorithm_id`, so this event run lands on the *same* bundle as its vector counterpart.


In [ ]:
from investing_algorithm_framework import generate_algorithm_id
from investing_algorithm_framework.domain import tqdm

top_param_variations = []
skipped_ids = []
for bt in top_10_backtests:
    metadata = bt.get_metadata() or {}
    variant = dict(metadata.get("params") or {})
    if not variant:
        # Legacy bundles without metadata["params"] fall back to the
        # canonical ``Backtest.parameters`` slot.
        variant = dict(bt.parameters or {})
    if not variant:
        skipped_ids.append(bt.algorithm_id)
        continue
    top_param_variations.append(variant)

print(f"Recovered {len(top_param_variations)} param sets from {len(top_10_backtests)} bundles")
if skipped_ids:
    print(f"Skipped {len(skipped_ids)} bundle(s) with no saved params: {skipped_ids}")


def initialize_strategies(
    strategy_class,
    param_variations,
    symbols,
    market,
    trading_symbol="EUR",
):
    """Re-instantiate the in-sample winners for the event-engine run.

    Mirrors notebook 03: drop underscore-prefixed metadata (e.g.
    ``_grid_profile``) before hashing and before passing to the
    strategy constructor, so the recomputed ``algorithm_id`` matches
    the one the vector sweep already used.
    """
    strategies = []

    for variant in tqdm(
        param_variations, desc="Initializing strategies", colour="green"
    ):
        strategy_params = {
            k: v for k, v in variant.items() if not k.startswith("_")
        }

        strategy = strategy_class(
            algorithm_id=generate_algorithm_id(params=strategy_params),
            symbols=symbols,
            trading_symbol=trading_symbol,
            market=market,
            metadata={"params": variant, "symbols": symbols, "market": market},
            **strategy_params,
        )
        strategies.append(strategy)

    return strategies


top_10_strategies = initialize_strategies(
    strategy_class=Strategy,
    param_variations=top_param_variations,
    symbols=event_study.universe.symbols,
    market=event_study.universe.market,
    trading_symbol=event_study.universe.trading_symbol,
)


## Run the event backtest — merges into the SAME bundles

`app.run_backtest(..., backtest_storage_directory=top_selection_path)` re-saves each `<algorithm_id>.obtf` with `merge=True`: the new `event` slot is appended to the existing bundle, and the `vector` slot (all of notebook 03's rolling windows) is left untouched. No manual `PortfolioConfiguration` wiring is needed either — `event_study.universe`/`initial_capital` already fully describe the portfolio.


In [ ]:
from investing_algorithm_framework import create_app, RESOURCE_DIRECTORY, DATA_DIRECTORY

app = create_app(config={RESOURCE_DIRECTORY: "./resources", DATA_DIRECTORY: data_storage_path})

# Independent event algorithms run in isolated workers. Position sizing
# already follows the live simulated balance in the event engine.
event_backtests = app.run_backtest(
    strategies=top_10_strategies,
    study=event_study,
    continue_on_error=False,
    backtest_storage_directory=str(top_selection_path),
    n_workers=4,
    memory_budget_mb=16_384,
    min_available_memory_mb=4_096,
    show_progress=True,
)

print(f"\nEvent validation complete — {event_backtests.df['algorithm_id'].nunique()} backtests")


## Compare vector vs event for window 1

Same bundles, same study, two engines. If a strategy's vector-engine window-1 numbers don't roughly hold up here, it's not worth carrying into the out-of-sample notebooks.


In [ ]:
from investing_algorithm_framework import (
    DEFAULT_TRADE_METRIC_COLUMNS,
    get_backtests,
    show_backtest_runs,
    show_backtest_summaries,
)

# Reload once more so both the vector slot (all rolling windows) and
# the event slot (just merged in) are visible on the same objects.
merged_backtests = get_backtests(str(top_selection_path), top_10_ids)

window_1 = event_study.backtest_windows[0]

for engine in ("vector", "event"):
    show_backtest_runs(
        merged_backtests,
        engine=engine,
        study=IN_SAMPLE_STUDY,
        columns=DEFAULT_TRADE_METRIC_COLUMNS,
        run=window_1,
        sort_by="profit_factor",
        page_size=TOP_N,
    )
